# Файнтюн чемпиона Qwen3-32B QLoRA в Google Colab

Прогон **одной** лучшей модели по итогам 3 раундов на сервере MEPhI с двумя свежими исправлениями:
1. **Class weights в loss** (`use_class_weights=true`) — главный путь к +3-7pt на macro_f1
2. **Train/val split 90/10 stratified** (`val_split=0.10`) — устраняет selection bias

**Что нужно:**
- Colab Pro / Pro+ с **A100-40GB** или **L4-24GB** (профиль L4 уже подготовлен)
- Папка на Google Drive `MyDrive/VKR/Data/` с CSV-данными

**Что сохранится на Drive:** адаптер, results CSV, manifest.json. Логи Trainer печатаются прямо в output ячейки.

## 1. Монтируем Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
from datetime import datetime

DRIVE_ROOT = Path('/content/drive/MyDrive/VKR')
DRIVE_DATA = DRIVE_ROOT / 'Data'
RUN_TS = datetime.now().strftime('%Y%m%d_%H%M%S')
DRIVE_RUN = DRIVE_ROOT / 'finetune_runs' / RUN_TS
DRIVE_RUN.mkdir(parents=True, exist_ok=True)

print(f'Drive root      : {DRIVE_ROOT} (exists={DRIVE_ROOT.exists()})')
print(f'Drive Data      : {DRIVE_DATA} (exists={DRIVE_DATA.exists()})')
print(f'Run output      : {DRIVE_RUN}')

assert DRIVE_DATA.exists(), (
    f'Папка с данными {DRIVE_DATA} не найдена. '
    'Положи на Drive: data_after_stage3.csv, data_test.csv, train_after_eda.csv.'
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive root      : /content/drive/MyDrive/VKR (exists=True)
Drive Data      : /content/drive/MyDrive/VKR/Data (exists=True)
Run output      : /content/drive/MyDrive/VKR/finetune_runs/20260430_123829


## 2. Клонируем / обновляем код из GitHub

In [2]:
# === Если репо ПРИВАТНЫЙ — раскомментируй и подставь PAT ===
# GITHUB_PAT = 'ghp_xxxxxxxxxxxxxxxxxxxx'
# REPO_URL = f'https://{GITHUB_PAT}@github.com/KVTur23/Mifi_VKR.git'

# === Публичный репо ===
REPO_URL = 'https://github.com/KVTur23/Mifi_VKR.git'
REPO_BRANCH = 're-augmentation'
REPO_DIR = Path('/content/VKR')

In [3]:
import subprocess

if (REPO_DIR / '.git').exists():
    print('Репо уже клонирован — git pull')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', REPO_BRANCH], check=True)
else:
    print(f'Клонирую {REPO_URL} в {REPO_DIR}')
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

commit = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'],
    capture_output=True, text=True, check=True,
).stdout.strip()
print(f'\nТекущий коммит: {commit}')

Репо уже клонирован — git pull

Текущий коммит: 8a75953 data


## 3. Подцепляем данные с Drive (symlink)

In [4]:
PROJECT_CODE = REPO_DIR / 'code'
LOCAL_DATA = PROJECT_CODE / 'Data'

# Если в репо есть пустая Data/ — переименовываем, чтобы не конфликтовала
if LOCAL_DATA.exists() and not LOCAL_DATA.is_symlink():
    backup = LOCAL_DATA.parent / 'Data_repo_orig'
    if not backup.exists():
        LOCAL_DATA.rename(backup)
    else:
        import shutil; shutil.rmtree(LOCAL_DATA)

LOCAL_DATA.mkdir(exist_ok=True)
for csv in DRIVE_DATA.glob('*.csv'):
    target = LOCAL_DATA / csv.name
    if not target.exists():
        target.symlink_to(csv)

(LOCAL_DATA / 'finetune_checkpoints').mkdir(exist_ok=True)
(PROJECT_CODE / 'results').mkdir(exist_ok=True)

print('Data в репо:')
for f in sorted(LOCAL_DATA.iterdir()):
    if f.is_symlink():
        print(f'  {f.name} -> {f.readlink()}')
    elif f.is_dir():
        print(f'  {f.name}/ ({len(list(f.iterdir()))} items)')

Data в репо:
  _stage3_pairs_cache.bak.csv -> /content/drive/MyDrive/VKR/Data/_stage3_pairs_cache.bak.csv
  _stage3_pairs_cache_deu_Latn.bak.csv -> /content/drive/MyDrive/VKR/Data/_stage3_pairs_cache_deu_Latn.bak.csv
  _stage3_pairs_cache_eng_Latn.bak.csv -> /content/drive/MyDrive/VKR/Data/_stage3_pairs_cache_eng_Latn.bak.csv
  _stage3_pairs_cache_fra_Latn.bak.csv -> /content/drive/MyDrive/VKR/Data/_stage3_pairs_cache_fra_Latn.bak.csv
  _stage3_pairs_cache_round1_deu_Latn.bak.csv -> /content/drive/MyDrive/VKR/Data/_stage3_pairs_cache_round1_deu_Latn.bak.csv
  _stage3_pairs_cache_round1_eng_Latn.bak.csv -> /content/drive/MyDrive/VKR/Data/_stage3_pairs_cache_round1_eng_Latn.bak.csv
  _stage3_pairs_cache_round1_fra_Latn.bak.csv -> /content/drive/MyDrive/VKR/Data/_stage3_pairs_cache_round1_fra_Latn.bak.csv
  _stage3_pairs_cache_round2_deu_Latn.bak.csv -> /content/drive/MyDrive/VKR/Data/_stage3_pairs_cache_round2_deu_Latn.bak.csv
  _stage3_pairs_cache_round2_eng_Latn.bak.csv -> /content/dri

## 4. Устанавливаем зависимости

In [5]:
!pip install -q transformers==4.57.6 peft bitsandbytes accelerate datasets sentencepiece scikit-learn pandas filelock


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 115.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 38.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.7 MB/s eta 0:00:00


## 5. Проверка GPU + sys.path

In [5]:
import torch
import sys

if str(PROJECT_CODE) not in sys.path:
    sys.path.insert(0, str(PROJECT_CODE))

if not torch.cuda.is_available():
    raise RuntimeError('CUDA недоступна. Runtime → Change runtime type → A100/L4 GPU')

name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}  ({vram_gb:.1f} GB)')

if vram_gb >= 75:
    GPU_PROFILE = 'A100_80'
elif vram_gb >= 38:
    GPU_PROFILE = 'A100_40'
elif vram_gb >= 22:
    GPU_PROFILE = 'L4'
    print('⚠ L4 24GB — впритык, но профиль уже подготовлен (batch=1, max_seq=1536).')
else:
    raise RuntimeError(f'GPU слишком маленькая ({vram_gb:.1f} GB). Нужно ≥24GB.')

print(f'Профиль: {GPU_PROFILE}')

GPU: NVIDIA A100-SXM4-40GB  (42.4 GB)
Профиль: A100_40


## 6. Финализируем конфиг — class_weights + val_split

In [6]:
import json, shutil

config_path = PROJECT_CODE / 'config_models' / 'finetune_configs' / 'qwen3_32b_qlora.json'

with open(config_path) as f:
    cfg = json.load(f)

# === Базовые фиксы (всегда вкл) ===
cfg['use_class_weights'] = True
cfg['val_split'] = 0.10

# === LoRA rank ===
# Прошлые раны: r=32 (recovered E2 macro_f1=0.530) ≈ r=16 (final 0.524).
# Различия в шуме. Возвращаем r=32 как более стабильный.
# cfg['peft_config']['r'] = 32
# cfg['peft_config']['lora_alpha'] = 64

# === Loss type: 'ce' (default) | 'focal' ===
# Focal loss даунвейтит "лёгкие" примеры, фокусируется на hard examples.
# Совместим с class_weights (alpha_t из весов).
# Пример включения:
# cfg['loss'] = 'focal'
# cfg['focal_gamma'] = 2.0   # стандартное значение из оригинальной статьи

# === Hierarchy regularizer ===
# Штраф за prob mass на классах из неправильной группы (A/B/C).
# Заставляет модель сначала научиться различать группы, потом класс
# внутри группы. Цель: пробить f1_C ceiling=0.533.
# cfg['use_hierarchy'] = True
# cfg['hierarchy_lambda'] = 0.3

# === Per-epoch test eval ===
# Прогоняет TEST после каждой эпохи, пишет в test_curve.csv.
# НЕ влияет на выбор best checkpoint (val остаётся главным селектором),
# нужно для диагностики: где реальный пик на test, а не на val.
# Стоимость: ~2-3 мин на эпоху.
cfg['eval_test_each_epoch'] = True

with open(config_path, 'w') as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)

shutil.copy(config_path, DRIVE_RUN / 'config_used.json')

print('=== config ===')
print(json.dumps(cfg, ensure_ascii=False, indent=2))


=== config ===
{
  "model_name": "Qwen/Qwen3-32B",
  "method": "qlora",
  "task_type": "SEQ_CLS",
  "num_labels": 36,
  "max_seq_length": 2048,
  "peft_config": {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "bias": "none",
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "modules_to_save": [
      "score"
    ]
  },
  "quantization": {
    "load_in_4bit": true,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_use_double_quant": true
  },
  "training_params": {
    "num_train_epochs": 5,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 8,
    "learning_rate": 0.0002,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.03,
    "weight_decay": 0.01,
    "optim": "adamw_torch",
    "gradient_checkpointing": true,
    "max_grad_norm": 1.0,
    "logging_steps": 10

## 7. Sanity-check + cd

In [7]:
%cd {PROJECT_CODE}

from src.finetune.orchestrator import METHOD_CONFIGS, run_finetune
from src.finetune.run_qlora import run as run_qlora_check

cp = METHOD_CONFIGS['qlora_qwen3_32b']
print(f'Конфиг path: {cp}')
print(f'Существует:  {cp.exists()}')
print(f'run_qlora импортирован: ✓')

import pandas as pd
results_csv = PROJECT_CODE / 'results' / 'finetune_results.csv'
if results_csv.exists():
    print(f'\n⚠ Уже есть {results_csv}:')
    print(pd.read_csv(results_csv).to_string(index=False))
    print('→ ниже запустим с force=True')
else:
    print(f'\nfinetune_results.csv ещё не создан — чистый прогон')

/content/VKR/code
Конфиг path: /content/VKR/code/config_models/finetune_configs/qwen3_32b_qlora.json
Существует:  True
run_qlora импортирован: ✓

finetune_results.csv ещё не создан — чистый прогон


## 8. Прогон тренировки

**Время**: A100-40 ~3 ч, L4 ~5 ч. Логи Trainer (loss / eval_balanced_accuracy / eval_macro_f1 по эпохам) пойдут прямо в output ячейки — можно следить.

In [8]:
df = run_finetune(
    methods=['qlora_qwen3_32b'],
    gpu=GPU_PROFILE,
    force=True,
)
df

[Orchestrator] GPU профиль: A100_40
[Orchestrator] Методы: ['qlora_qwen3_32b']
[Orchestrator] force=True, уже готово: —

############################################################
# qlora_qwen3_32b  (run_key=qlora_qwen3_32b)
############################################################
[Данные] Найден чекпоинт этапа 3: data_after_stage3.csv (2331 записей)
[Данные] Тестовая выборка: data_test.csv (341 записей)
[Данные] Найден чекпоинт этапа 0: train_after_eda.csv (1409 записей)
[ClassWeights] enabled. range=[0.329, 1.408], mean=1.156
[Split] train=2097 | val=234 (val_split=0.1) | test=341 (held out)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Epoch,Training Loss,Validation Loss,Balanced Accuracy,Macro F1
1,12.583000,1.430132,0.625255,0.602894
2,5.822200,0.774722,0.803342,0.774888
3,1.541000,0.974637,0.802603,0.796028
4,0.027200,1.146557,0.830613,0.815938
5,0.004600,1.170137,0.832497,0.819321


[TestEval] E1.0: bal_acc=0.4219, macro_f1=0.3768, f1_A=0.4744, f1_B=0.4802, f1_C=0.4222
[TestEval] E2.0: bal_acc=0.5106, macro_f1=0.4472, f1_A=0.7178, f1_B=0.5680, f1_C=0.4222
[TestEval] E3.0: bal_acc=0.4800, macro_f1=0.4610, f1_A=0.7053, f1_B=0.5687, f1_C=0.3667
[TestEval] E4.0: bal_acc=0.5227, macro_f1=0.4964, f1_A=0.7309, f1_B=0.5412, f1_C=0.5056
[TestEval] E5.0: bal_acc=0.5352, macro_f1=0.5031, f1_A=0.7331, f1_B=0.5806, f1_C=0.5056
[Данные] Тестовая выборка: data_test.csv (341 записей)


/content/VKR/code/src/finetune/evaluate_finetuned.py:134: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([row], columns=RESULTS_COLUMNS)], ignore_index=True)


FINETUNE EVAL: qlora_qwen3_32b
  Balanced Accuracy: 0.5352
  Macro F1:          0.5031
  F1 Group A:        0.7331
  F1 Group B:        0.5806
  F1 Group C:        0.5056

                                                                              precision    recall  f1-score   support

                                                       Блок бизнес-директора       0.50      1.00      0.67         1
                                                      Блок деректора по газу       0.35      0.43      0.39        14
                                          Блок директора по газовым проектам       1.00      0.17      0.29         6
                                                 Блок директора по мощностям       0.83      0.79      0.81        48
                                                 Блок директора по персоналу       1.00      0.33      0.50         3
                                                  Блок директора по портфелю       0.25      0.25      0.25         4
 

,run_key,method,model,balanced_accuracy,macro_f1,f1_group_A,f1_group_B,f1_group_C,trainable_params,train_time_sec,timestamp
0,qlora_qwen3_32b,qlora,Qwen/Qwen3-32B,0.535181,0.503085,0.733064,0.580635,0.505556,134402048,16528.690352,2026-04-30T17:25:52


## 9. Сохраняем адаптер + results на Drive

In [9]:
import shutil

# 1. Адаптер
src_adapter = PROJECT_CODE / 'Data' / 'finetune_checkpoints' / 'qlora_qwen3_32b'
dst_adapter = DRIVE_RUN / 'adapter'
if src_adapter.exists():
    print(f'Копирую адаптер → Drive (1-3 минуты)...')
    shutil.copytree(src_adapter, dst_adapter, dirs_exist_ok=True)
    sz = sum(f.stat().st_size for f in dst_adapter.rglob('*') if f.is_file()) / 1e6
    print(f'✓ Адаптер: {sz:.1f} МБ')
else:
    print(f'⚠ Адаптер не найден: {src_adapter}')

# 2a. Test curve CSV (per-epoch test metrics) — копируем на Drive если есть
test_curve = src_adapter / 'test_curve.csv' if src_adapter.exists() else None
if test_curve and test_curve.exists():
    shutil.copy(test_curve, dst_adapter / 'test_curve.csv')
    print(f'✓ Test curve → {dst_adapter / "test_curve.csv"}')

# 2. Results CSV + per-sample preds
src_results = PROJECT_CODE / 'results'
dst_results = DRIVE_RUN / 'results'
shutil.copytree(src_results, dst_results, dirs_exist_ok=True)
print(f'✓ Results → {dst_results}')

# 3. Manifest для воспроизводимости
manifest = {
    'timestamp': RUN_TS,
    'gpu': name,
    'gpu_profile': GPU_PROFILE,
    'commit': commit,
    'method': 'qlora_qwen3_32b',
    'config': cfg,
    'metrics': df.to_dict('records') if not df.empty else [],
}
with open(DRIVE_RUN / 'manifest.json', 'w') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print(f'✓ Manifest → {DRIVE_RUN / "manifest.json"}')

Копирую адаптер → Drive (1-3 минуты)...
✓ Адаптер: 1660.9 МБ
✓ Test curve → /content/drive/MyDrive/VKR/finetune_runs/20260430_123829/adapter/test_curve.csv
✓ Results → /content/drive/MyDrive/VKR/finetune_runs/20260430_123829/results
✓ Manifest → /content/drive/MyDrive/VKR/finetune_runs/20260430_123829/manifest.json


## 10. Финальные метрики + сравнение с baseline R1

In [ ]:
import pandas as pd

results_csv = PROJECT_CODE / 'results' / 'finetune_results.csv'
if results_csv.exists():
    df_full = pd.read_csv(results_csv)
    print('=== finetune_results.csv ===\n')
    print(df_full.to_string(index=False))

    print('\n=== Сравнение с baseline (R1 без фиксов) ===')
    print('R1 baseline qlora_qwen3_32b: bal_acc=0.5615, macro_f1=0.5228, f1_C=0.625')
    if 'qlora_qwen3_32b' in df_full['run_key'].values:
        row = df_full[df_full['run_key'] == 'qlora_qwen3_32b'].iloc[0]
        print(f'Текущий прогон:              bal_acc={row["balanced_accuracy"]:.4f}, '
              f'macro_f1={row["macro_f1"]:.4f}, f1_C={row["f1_group_C"]:.4f}')
        print(f'\nΔ bal_acc:  {row["balanced_accuracy"] - 0.5615:+.4f}')
        print(f'Δ macro_f1: {row["macro_f1"] - 0.5228:+.4f}')
        print('\nNB: baseline R1 имел selection bias (~+1-2pt вверх).')
        print('Текущая цифра честная — split 90/10 с val-set.')
else:
    print('⚠ finetune_results.csv не найден')

---
## Diagnostic — если ячейка 8 завершилась слишком быстро (<1 мин)

Запусти эту ячейку чтобы посмотреть, что не так:

In [ ]:
import traceback

print('=== 1. CSV с результатами ===')
if results_csv.exists():
    print(pd.read_csv(results_csv))
else:
    print('не существует')

print('\n=== 2. Конфиг находится? ===')
cp = METHOD_CONFIGS['qlora_qwen3_32b']
print(f'  Path:    {cp}')
print(f'  Exists:  {cp.exists()}')

print('\n=== 3. Импорт run_qlora ===')
try:
    from src.finetune.run_qlora import run
    print('  ✓ OK')
except Exception as e:
    print(f'  ✗ {type(e).__name__}: {e}')
    traceback.print_exc()

print('\n=== 4. Прямой вызов run() — обходим try/except orchestrator ===')
try:
    from src.finetune.orchestrator import _load_pipeline_cfg
    pipeline_cfg = _load_pipeline_cfg(GPU_PROFILE)
    print(f'  pipeline_cfg loaded: gpu={pipeline_cfg["gpu_name"]}')

    from src.finetune.run_qlora import run
    print(f'  Запускаю run() напрямую...')
    run(config_path=str(cp), pipeline_cfg=pipeline_cfg)
except Exception as e:
    print(f'  ✗ {type(e).__name__}: {e}')
    traceback.print_exc()

## 11. Eval восстановленных адаптеров (recovery после краша 2026-04-29)

Краш на сохранении `checkpoint-396` забил оба диска (Colab + Drive). Адаптеры спасены вручную:
`MyDrive/VKR/recovered_weights/qlora_qwen3_32b_e{2,3}/`.

- **E2** (epoch 2, был best by val macro_f1 = 0.787)
- **E3** (epoch 3, val метрик нет — краш до eval-фазы)

Ниже — гонка обоих адаптеров на тесте через `evaluate_finetuned.evaluate()`.


In [ ]:
# Sanity-check восстановленных адаптеров на Drive
RECOVERED_DIR = DRIVE_ROOT / 'recovered_weights'
ADAPTER_E2 = RECOVERED_DIR / 'qlora_qwen3_32b_e2'
ADAPTER_E3 = RECOVERED_DIR / 'qlora_qwen3_32b_e3'

assert ADAPTER_E2.exists(), f'Не найдено: {ADAPTER_E2}'
assert ADAPTER_E3.exists(), f'Не найдено: {ADAPTER_E3}'

REQUIRED = [
    'adapter_model.safetensors', 'adapter_config.json',
    'id2label.json', 'class_groups.json',
    'tokenizer.json', 'tokenizer_config.json',
]
for d in (ADAPTER_E2, ADAPTER_E3):
    missing = [f for f in REQUIRED if not (d / f).exists()]
    assert not missing, f'{d.name}: не хватает {missing}'
    sz = (d / 'adapter_model.safetensors').stat().st_size / 1e9
    print(f'  {d.name}: adapter={sz:.2f} GB ✓')


In [ ]:
# Eval E2 + E3 → results/finetune_results.csv + per-sample preds
import gc, json, os, shutil
import torch
import pandas as pd

# CWD = PROJECT_CODE — иначе evaluate пишет в неправильное место (relative paths)
os.chdir(PROJECT_CODE)

with open(PROJECT_CODE / 'config_models' / 'pipeline_config.json') as f:
    pipeline_cfg = json.load(f)
pipeline_cfg['gpu_name'] = GPU_PROFILE

CONFIG_PATH = PROJECT_CODE / 'config_models' / 'finetune_configs' / 'qwen3_32b_qlora.json'

from src.finetune.evaluate_finetuned import evaluate

results = {}
for tag, adapter_dir in [('e2', ADAPTER_E2), ('e3', ADAPTER_E3)]:
    run_key = f'qlora_qwen3_32b_recovered_{tag}'
    print(f'\n{"="*60}\nEvaluate {tag.upper()}: {adapter_dir.name}\n{"="*60}')
    res = evaluate(
        adapter_dir=str(adapter_dir),
        config_path=str(CONFIG_PATH),
        pipeline_cfg=pipeline_cfg,
        run_key=run_key,
    )
    results[tag] = res

    # Освобождаем VRAM перед загрузкой следующего base+adapter
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def _fmt(v):
    return round(v, 4) if v is not None else None

summary = pd.DataFrame([
    {
        'epoch': k.upper(),
        'bal_acc':  _fmt(v['balanced_accuracy']),
        'macro_f1': _fmt(v['macro_f1']),
        'f1_A': _fmt(v['f1_group_A']),
        'f1_B': _fmt(v['f1_group_B']),
        'f1_C': _fmt(v['f1_group_C']),
    }
    for k, v in results.items()
])
print('\n=== Сводка восстановленных метрик ===')
print(summary.to_string(index=False))

# Backup на Drive — чтобы не потерять снова
RECOVERED_OUT = DRIVE_ROOT / 'recovered_results'
RECOVERED_OUT.mkdir(exist_ok=True)
summary.to_csv(RECOVERED_OUT / 'qlora_qwen3_32b_recovered_summary.csv', index=False)
for tag in ('e2', 'e3'):
    p = PROJECT_CODE / 'results' / f'preds_qlora_qwen3_32b_recovered_{tag}.csv'
    if p.exists():
        shutil.copy(p, RECOVERED_OUT / p.name)
results_csv = PROJECT_CODE / 'results' / 'finetune_results.csv'
if results_csv.exists():
    shutil.copy(results_csv, RECOVERED_OUT / 'finetune_results.csv')
print(f'\nBackup → {RECOVERED_OUT}')
